In [1]:
# varify directories and configurations availibility

import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

OPENAI_API_KEY=os.getenv("OPENAI_API_KEY")

PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "data"
AUDIO_OUTPUT_DIR = DATA_DIR / "input" / "audio"
SAMPLE_DATA_DIR = DATA_DIR / "sample_data"

AUDIO_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SAMPLE_DATA_DIR.mkdir(parents=True, exist_ok=True)

client = OpenAI(api_key=OPENAI_API_KEY)

print(f"Project Root        : {PROJECT_ROOT}")
print(f"Audio Directory     : {AUDIO_OUTPUT_DIR}")
print(f"Sample Data Dir     : {SAMPLE_DATA_DIR}")

Project Root        : d:\LearnPython\multi-model-capability-project
Audio Directory     : d:\LearnPython\multi-model-capability-project\data\input\audio
Sample Data Dir     : d:\LearnPython\multi-model-capability-project\data\sample_data


In [2]:
# prompt and scenarios for audio generation
TRANSCRIPT_PROMPT = """
You are an expert customer support conversation simulator.

Generate a realistic conversation between a customer and customer support agent.

Scenario: {scenario}

Requirements:
- Conversation duration should be approximately 1-2 minutes when spoken aloud.
- Include natural greetings and a professional closing.
- The customer should clearly explain their issue.
- The support agent should ask relevant questions.
- The support agent should provide an appropriate resolution whenever possible.
- Make the conversation realistic, random sentiment, and conversational.
- Do NOT include speaker timestamps.
- Prefix each utterance with either "Customer:" or "Agent:".
- Return ONLY the conversation transcript.
"""

SCENARIOS =[
    "Billing Issue",
    "Product Technical Support",
    "Product Inquiry",
    "Refund not credited",
    "Product quality issue",
    "Subscription cancellation",
    "Delivery delay"
]

print("Prompt template created successfully.")
print(f"Total scenarios : {len(SCENARIOS)}")


Prompt template created successfully.
Total scenarios : 7


In [ ]:
# Generate Synthetic Customer COnversations

import random

NUM_CONVERSATIONS = 5
records = []

for conversation_id in range(1, NUM_CONVERSATIONS +1):

    scenario = random.choice(SCENARIOS)
    print(f"Generating Conversation {conversation_id} | Scenario: {scenario}")

    response = client.responses.create(
        model="gpt-4.1-mini",
        input=TRANSCRIPT_PROMPT.format(
            scenario=scenario
        )
    )

    transcript = response.output_text

    records.append(
        {
            "conversation_id": conversation_id,
            "scenario": scenario,
            "transcript": transcript
        }
    )

    print("="*80)
    print(transcript)
    print("="*80)


conversation_df = pd.DataFrame(records)
print("\n Dataset generated successfully.")

conversation_df.head()

Generating Conversation 1 | Scenario: Product quality issue
Customer: Hi, I recently bought one of your wireless headphones, and I'm having some problems with the sound quality.

Agent: Hello! I'm sorry to hear that. Can you tell me a bit more about the issue you're experiencing?

Customer: Yeah, sure. The sound keeps cutting out randomly, and sometimes there's this annoying static noise. It's really frustrating since I just got them last week.

Agent: That does sound frustrating. Just to clarify, is the issue happening with both earbuds or just one?

Customer: It's mostly happening in the right earbud, but occasionally in the left one too.

Agent: Thanks for that detail. Are you using them with any specific device, like a phone or a laptop?

Customer: Mostly with my phone, but I tried them with my laptop and noticed the same problem.

Agent: Got it. Have you tried resetting the headphones or updating their firmware? Sometimes that can help with connectivity issues.

Customer: I tried 

In [8]:
# save dataset

conversation_df["audio_file_name"] = [
    f"call_{row.scenario}.wav"
    for index, row in conversation_df.iterrows()
]

conversation_df["audio_generated"] = False

csv_path = SAMPLE_DATA_DIR / "customer_conversations.csv"

conversation_df.to_csv(
    csv_path,
    index=False
)

print("Conversation dataset saved successfully.")
conversation_df

Conversation dataset saved successfully.


,conversation_id,scenario,transcript,audio_file_name,audio_generated
0,1,Product quality issue,"Customer: Hi, I recently bought one of your wi...",call_Product quality issue.wav,False
1,2,Product Technical Support,"Customer: Hi there, I recently bought your wir...",call_Product Technical Support.wav,False
2,3,Product Inquiry,"Customer: Hi, good afternoon. I was hoping you...",call_Product Inquiry.wav,False
3,4,Billing Issue,"Customer: Hi there, I hope you can help me. I ...",call_Billing Issue.wav,False
4,5,Subscription cancellation,"Customer: Hi there, I hope you can help me. I’...",call_Subscription cancellation.wav,False


In [9]:
# generate audio files
from pathlib import Path

voice = "alloy"

for index, row in conversation_df.iterrows():

    transcript = row["transcript"]
    audio_file = AUDIO_OUTPUT_DIR / row["audio_file_name"]

    print(f"Generating: {audio_file.name}")

    with client.audio.speech.with_streaming_response.create(
        model="gpt-4o-mini-tts",
        voice=voice,
        input=transcript,
        response_format="wav"
    ) as response:
        response.stream_to_file(audio_file)

    conversation_df.loc[index, "audio_generated"] = True

print("All audio files generated successfully.")

conversation_df

Generating: call_Product quality issue.wav
Generating: call_Product Technical Support.wav
Generating: call_Product Inquiry.wav
Generating: call_Billing Issue.wav
Generating: call_Subscription cancellation.wav
All audio files generated successfully.


,conversation_id,scenario,transcript,audio_file_name,audio_generated
0,1,Product quality issue,"Customer: Hi, I recently bought one of your wi...",call_Product quality issue.wav,True
1,2,Product Technical Support,"Customer: Hi there, I recently bought your wir...",call_Product Technical Support.wav,True
2,3,Product Inquiry,"Customer: Hi, good afternoon. I was hoping you...",call_Product Inquiry.wav,True
3,4,Billing Issue,"Customer: Hi there, I hope you can help me. I ...",call_Billing Issue.wav,True
4,5,Subscription cancellation,"Customer: Hi there, I hope you can help me. I’...",call_Subscription cancellation.wav,True
